In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA


In [2]:
df=pd.read_csv("01 covid_toy.csv")

display(df.head())


,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [3]:
numeric_columns=df.select_dtypes(include=np.number).columns

print("Numeric Colums:")
print(numeric_columns)

Numeric Colums:
Index(['age', 'fever'], dtype='object')


In [4]:
categorical_colums=df.select_dtypes(include="object").columns

print("Categorical Colums:")
print(categorical_colums)


Categorical Colums:
Index(['gender', 'cough', 'city', 'has_covid'], dtype='object')


In [9]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        100 non-null    int64  
 1   gender     100 non-null    object 
 2   fever      90 non-null     float64
 3   cough      100 non-null    object 
 4   city       100 non-null    object 
 5   has_covid  100 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 4.8+ KB


In [11]:
df["gender"] = df["gender"].astype("object")
df["cough"] = df["cough"].astype("object")
df["city"] = df["city"].astype("object")
df["has_covid"] = df["has_covid"].astype("object")

In [13]:
df["fever"] = df["fever"].fillna(df["fever"].mean())

In [14]:
df["fever"] = df["fever"].fillna(df["fever"].median())

In [15]:
df["gender"] = df["gender"].fillna(df["gender"].mode()[0])

In [16]:
df["city"] = df["city"].fillna("Unknown")

In [17]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[["age", "fever"]] = scaler.fit_transform(df[["age", "fever"]])

In [18]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df[["age", "fever"]] = scaler.fit_transform(df[["age", "fever"]])

In [19]:
df.isnull().sum()

,0
age,0
gender,0
fever,0
cough,0
city,0
has_covid,0


In [22]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler().set_output(transform="pandas")

numeric_cols = df.select_dtypes(include=["number"]).columns

scaled_data = scaler.fit_transform(df[numeric_cols])

scaled_data.head()

,age,fever
0,0.696203,0.833333
1,0.278481,0.333333
2,0.468354,0.500000
3,0.329114,0.000000
4,0.759494,0.500000


In [23]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().set_output(transform="pandas")

numeric_cols = df.select_dtypes(include=["number"]).columns

standardized_data = scaler.fit_transform(df[numeric_cols])

standardized_data.head()

,age,fever
0,0.637467,1.111906
1,-0.695639,-0.435592
2,-0.089682,0.080241
3,-0.534050,-1.467257
4,0.839453,0.080241


In [24]:
categorical_cols = df.select_dtypes(include=["object"]).columns

df[categorical_cols].head()

,gender,cough,city,has_covid
0,Male,Mild,Kolkata,No
1,Male,Mild,Delhi,Yes
2,Male,Mild,Delhi,No
3,Female,Mild,Kolkata,No
4,Female,Mild,Mumbai,No


In [27]:
from sklearn.feature_selection import SelectKBest, f_regression

In [29]:
y = df["has_covid"]

In [30]:
numeric_features = scaled_data.drop(
    columns=["has_covid"], errors="ignore"
)

In [31]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
).set_output(transform="pandas")

categorical_cols = df.select_dtypes(
    include=["object", "category"]
).drop(columns=["has_covid"], errors="ignore").columns

encoded_data = encoder.fit_transform(df[categorical_cols])

In [32]:
X = pd.concat(
    [numeric_features, encoded_data],
    axis=1
)

X_numeric = X.select_dtypes(include=np.number)

In [33]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(
    score_func=f_classif,
    k=3
).set_output(transform="pandas")

X_selected = selector.fit_transform(X_numeric, y)

print(X_selected.head())

      fever  cough_Mild  cough_Strong
0  0.833333         1.0           0.0
1  0.333333         1.0           0.0
2  0.500000         1.0           0.0
3  0.000000         1.0           0.0
4  0.500000         1.0           0.0
